# Replaying a Month of Incidents Through Four Alerting Rules

Companion notebook for the [Alerting on Symptoms](https://codemia.io/courses/observability_and_delivery/obs_alerting_on_symptoms) lesson on Codemia.

**What you will build**: a month of minute by minute traffic with five real incidents buried in it, from a ninety second blip
to a three day leak. You will run four alerting rules over the same data, a static threshold, a
single burn rate, the multi window table from the SRE workbook and a slow ticket rule, and measure
what each one catches, how fast, and how often it wakes someone for nothing.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(99)

DAYS = 30
MINUTES = DAYS * 24 * 60
SLO = 0.999                      # availability objective over the 30 day window
BUDGET = 1 - SLO                 # the fraction of requests allowed to fail

## Concept recap

An error budget turns an objective into a quantity. At 99.9 percent over thirty days you are allowed
to fail one request in a thousand, and the only question an alert has to answer is whether you are
spending that allowance faster than thirty days' worth.

The ratio of actual spend rate to allowed spend rate is the burn rate. A burn rate of one exhausts
the budget exactly at the end of the window. A burn rate of fourteen point four exhausts it in two
days, which is where that strange looking constant in every runbook comes from: it is thirty divided
by two, near enough.

An alert is a window and a threshold on that burn rate, and the window is the interesting half. A
short window notices a sharp failure quickly and fires on noise; a long window is immune to noise
and sleeps through most of an outage. The rest of this notebook is the measurement of that trade.

## A month of traffic, with five incidents in it

In [ ]:
minute = np.arange(MINUTES)
hour_of_day = (minute // 60) % 24

# Diurnal traffic: a quiet night, a busy afternoon, lighter at weekends.
diurnal = 0.45 + 0.55 * np.sin((hour_of_day - 4) / 24 * 2 * np.pi) ** 2
weekday = np.where(((minute // 1440) % 7) < 5, 1.0, 0.65)
requests = (2_400 * diurnal * weekday).astype(int)

BACKGROUND = 0.0002             # 0.02 percent of requests fail on an ordinary day
error_rate = np.full(MINUTES, BACKGROUND)

INCIDENTS = [
    ('hard outage', 3 * 1440 + 120, 20, 0.55),          # a config push, caught by a human in 20 min
    ('bad canary', 9 * 1440 + 840, 120, 0.030),         # one of twenty pods serving errors
    ('memory leak', 17 * 1440, 3 * 1440, 0.0040),       # three days of slightly elevated failures
    ('night blip', 22 * 1440 + 210, 1, 0.30),           # one minute, at 03:30, almost no traffic
    ('retry storm', 26 * 1440 + 1140, 45, 0.120),       # a dependency timing out under load
]
for _, start, length, rate in INCIDENTS:
    error_rate[start:start + length] = rate

errors = rng.binomial(requests, error_rate)

print(f'{"incident":>14}{"day":>6}{"length":>10}{"error rate":>12}'
      f'{"failed requests":>18}{"budget":>9}{"burn rate":>12}')
allowance = requests.sum() * BUDGET
for name, start, length, rate in INCIDENTS:
    bad = errors[start:start + length].sum()
    burn = bad / requests[start:start + length].sum() / BUDGET
    print(f'{name:>14}{start // 1440:>6}{length:>8} m{rate:>12.2%}'
          f'{bad:>18,}{bad / allowance:>9.1%}{burn:>11.0f}x')
print()
print(f'requests in the month      {requests.sum():>12,}')
print(f'failures in the month      {errors.sum():>12,}')
print(f'error budget for the month {allowance:>12,.0f} failed requests')
print(f'budget spent               {errors.sum() / allowance:>12.1%}')
print(f'spent on background noise  '
      f'{(errors.sum() - sum(errors[s:s + l].sum() for _, s, l, _ in INCIDENTS)) / allowance:>12.1%}')

Look at the last two columns before looking at any alerting. The night blip has the highest burn
rate in the table by a wide margin and spends almost none of the budget, because it lasted a minute
at the quietest hour of the day. The memory leak has the lowest burn rate of the five and is the most
expensive thing that happened all month, because it ran for three days and never looked bad enough
for anyone to investigate.

That inversion is the argument for burn rate alerting, and also its warning label. Severity as a
human reads it off a graph, how tall the spike is, is close to unrelated to the thing the objective
measures, which is how much of the allowance went away. A rule has to weigh rate against duration,
which is exactly what a burn rate over a window does.

In [ ]:
BIN = 10
binned = minute[::BIN]
ratio = (errors[:len(binned) * BIN].reshape(-1, BIN).sum(axis=1)
         / requests[:len(binned) * BIN].reshape(-1, BIN).sum(axis=1))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax1.plot(binned / 1440, np.maximum(ratio, 1e-6), linewidth=0.7, color='tab:red')
ax1.axhline(BUDGET, color='black', linestyle='--', label='the objective (0.1% errors)')
for name, start, length, _ in INCIDENTS:
    ax1.axvspan(start / 1440, (start + length) / 1440, color='tab:orange', alpha=0.25)
ax1.set_yscale('log')
ax1.set_ylabel('error ratio, 10 minute bins')
ax1.set_title('Five incidents. The flattest band is the most expensive one.')
ax1.legend()
ax1.grid(alpha=0.3)

spent = np.cumsum(errors) / allowance
ax2.plot(minute / 1440, spent * 100, color='tab:blue')
ax2.axhline(100, color='black', linestyle='--', label='budget exhausted')
ax2.plot(minute / 1440, minute / MINUTES * 100, color='grey', linestyle=':',
         label='burn rate of exactly 1')
ax2.set_xlabel('day of the month')
ax2.set_ylabel('error budget spent (%)')
ax2.legend()
ax2.grid(alpha=0.3)
plt.show()

The lower panel is the view an SLO gives you that a dashboard does not. The grey line is what
spending looks like if you fail exactly as much as you are allowed to; the blue line is what
happened. Every incident is a step, the leak is a ramp steeper than the grey line, and the question
an alert should answer is whether the blue line is currently steeper than the grey one.

## Four rules over the same data

In [ ]:
cum_err = np.concatenate([[0], np.cumsum(errors)])
cum_req = np.concatenate([[0], np.cumsum(requests)])


def window_burn(window_min):
    """Burn rate over a trailing window, per minute, as a multiple of the allowed rate."""
    bad = cum_err[window_min:] - cum_err[:-window_min]
    valid = cum_req[window_min:] - cum_req[:-window_min]
    burn = np.zeros(MINUTES)
    burn[window_min - 1:] = np.where(valid > 0, bad / np.maximum(valid, 1) / BUDGET, 0)
    return burn


def static_threshold(window_min, ratio_threshold):
    bad = cum_err[window_min:] - cum_err[:-window_min]
    valid = cum_req[window_min:] - cum_req[:-window_min]
    firing = np.zeros(MINUTES, dtype=bool)
    firing[window_min - 1:] = bad / np.maximum(valid, 1) > ratio_threshold
    return firing


def multi_window(rules):
    """Fire when a long window and its short window both exceed the factor."""
    firing = np.zeros(MINUTES, dtype=bool)
    for factor, long_min, short_min in rules:
        firing |= (window_burn(long_min) >= factor) & (window_burn(short_min) >= factor)
    return firing


PAGE_RULES = [(14.4, 60, 5), (6.0, 360, 30)]
TICKET_RULES = [(3.0, 1440, 120), (1.0, 4320, 360)]

STRATEGIES = {
    'static 5 min > 1% errors': static_threshold(5, 0.01),
    'single burn rate 14.4x over 1h': window_burn(60) >= 14.4,
    'multi window pages (14.4x/6x)': multi_window(PAGE_RULES),
    'multi window pages + tickets': multi_window(PAGE_RULES + TICKET_RULES),
}


def episodes(firing):
    """Distinct alert episodes: a rising edge starts one."""
    return np.flatnonzero(firing & ~np.concatenate([[False], firing[:-1]]))


def evaluate(firing, grace=30):
    """Count episodes, split them into real and spurious, and time the detections."""
    windows = [(s, s + l) for _, s, l, _ in INCIDENTS]
    starts = episodes(firing)

    outside = sum(1 for e in starts
                  if not any(a - grace <= e <= b + grace for a, b in windows))
    worst_flap = 0
    for a, b in windows:
        worst_flap = max(worst_flap, int(((starts >= a - grace) & (starts <= b + grace)).sum()))

    detection = []
    for name, start, length, _ in INCIDENTS:
        fired = np.flatnonzero(firing[start:start + length + 720])
        detection.append(int(fired[0]) if len(fired) else None)

    return {'episodes': len(starts), 'outside': outside, 'worst_flap': worst_flap,
            'minutes': int(firing.sum()), 'detection': detection}


print(f'{"strategy":>32}{"alerts":>8}{"with no incident":>18}'
      f'{"worst flap":>12}{"minutes firing":>16}')
results = {}
for name, firing in STRATEGIES.items():
    r = evaluate(firing)
    results[name] = r
    print(f'{name:>32}{r["episodes"]:>8}{r["outside"]:>18}'
          f'{r["worst_flap"]:>12}{r["minutes"]:>16,}')

In [ ]:
print('minutes from the start of each incident to the first alert')
print()
print(f'{"strategy":>32}' + ''.join(f'{n:>14}' for n, _, _, _ in INCIDENTS))
for name, r in results.items():
    cells = ''.join(f'{("never" if d is None else str(d) + " min"):>14}'
                    for d in r['detection'])
    print(f'{name:>32}' + cells)

Read the two tables together, because neither says anything on its own.

The static threshold is the fastest detector in the table and it pages for the night blip, a one
minute event that cost half a percent of the month. That is the behaviour that trains a team to
ignore its pager, and the training works: a pager that fires for one minute self healing events is
answered a little more slowly every month until nobody is really answering it.

The single fourteen point four rule is quiet and slower, and it sleeps through the leak completely.
Four tenths of a percent errors is a burn rate of four, and a rule looking for fourteen will never
see it however long it runs.

The two page rules together catch the three incidents that are worth waking up for, ignore the blip
and still miss the leak. Adding the ticket rules is what catches the leak, and the detection column
shows why those have to be tickets rather than pages: a window long enough to see a four times burn
takes most of a day to conclude anything, and there is nothing a woken human does with that
information at 04:00 that they would not do better at 09:00.

## Why the short window is there

In [ ]:
long_only = window_burn(360) >= 6.0
with_short = (window_burn(360) >= 6.0) & (window_burn(30) >= 6.0)

name, start, length, _ = INCIDENTS[4]            # the retry storm
print(f'incident: {name}, {length} minutes long')
print()
for label, firing in [('6x over 6h, long window only', long_only),
                      ('6x over 6h with a 30m short window', with_short)]:
    fired = np.flatnonzero(firing[start:start + 1440])
    detect = fired[0] if len(fired) else None
    still = np.flatnonzero(firing[start + length:start + length + 1440])
    linger = 0
    if len(still):
        run = still[0]
        while run + 1 < len(firing) and firing[start + length + run]:
            run += 1
            linger += 1
    print(f'{label}')
    print(f'  detected after        {detect} min')
    print(f'  still firing for      {linger} min after the incident ended')
    print(f'  total minutes firing  {firing.sum():,} in the month')
    print()

A six hour window keeps firing for hours after the failure stops, because the failure is still
inside the window. That is the lingering alert everybody has silenced manually at the end of an
incident, and it is why each rule in the workbook table carries a short window at one tenth the
length: the long window decides whether the burn is significant, and the short window decides whether
it is still happening.

The cost is a small amount of detection delay, which is almost always the right trade. An alert that
keeps ringing after the problem is fixed does not just annoy, it hides the next one.

## Picking a window and a factor, with the numbers in front of you

In [ ]:
print(f'{"window":>9}{"factor":>8}{"alerts":>8}{"flap":>6}'
      + ''.join(f'{n:>13}' for n, _, _, _ in INCIDENTS))
for window, factor in [(5, 14.4), (30, 14.4), (60, 14.4), (60, 6), (360, 6),
                       (360, 3), (1440, 3), (4320, 1)]:
    firing = (window_burn(window) >= factor) & (window_burn(max(window // 12, 1)) >= factor)
    r = evaluate(firing)
    cells = ''.join(f'{("never" if d is None else str(d) + " m"):>13}' for d in r['detection'])
    print(f'{window:>7} m{factor:>8}{r["episodes"]:>8}{r["worst_flap"]:>6}' + cells)
print()
print(f'time to exhaust a 30 day budget at each factor:')
for factor in [1, 3, 6, 14.4]:
    print(f'  {factor:>5}x -> {DAYS / factor:>5.1f} days')

The sweep is the whole design space on one screen, and it has a clear diagonal. Short windows with
high factors detect sharp failures in minutes and cannot see a slow burn at any duration. Long
windows with low factors see the slow burn and take between five and nineteen hours to say so. No
single row catches both a twenty minute outage and a three day leak, which is the entire reason the
multi window table exists: it is several rows of this sweep running at once.

The factors are not magic numbers either. They are the burn rates at which the budget runs out in two
days, five days, ten days and thirty days, and the last block of output is how you derive your own
for a different objective window.

Two honest caveats. First, nothing in this month fires spuriously at any window, because the
background floor is a clean 0.02 percent; the long windows earn their keep on services whose floor
is noisy, and that is the case this data cannot show you. Second, these are five synthetic incidents
with shapes I chose. Replay the same evaluation over your own history before committing to a factor,
because the right answer depends on the shape of your failures and the noisiness of your floor, both
of which are properties of your service rather than of the method.

## What to take away

- Convert the objective into a quantity first. The burn rate is the only number an alert needs, and
  it makes incidents comparable in the currency that matters.
- Visual severity and budget severity are different things. The spike nobody can miss is often
  cheap, and the slow degradation nobody notices is often the whole month's allowance.
- A static error ratio threshold pages on noise and still misses slow burns. It fails in both
  directions at once, which is why burn rate rules replaced it.
- Every rule needs a short confirmation window, or it keeps ringing long after the incident is over.
- Use several windows: fast and strict for outages, slow and gentle for leaks, and route the slow
  ones to a ticket queue because nothing useful happens when you wake somebody for a three day
  trend.